# 실제 도구로 ReAct 에이전트 만들기

이 노트북에서는 문제를 스스로 추론하고 여러 실제 도구 중에서 골라 문제를 해결하는 **ReAct 에이전트**를 만듭니다.

**ReAct**(Reasoning + Acting)는 단순하지만 강력합니다.
1. **추론(Reason)**: LLM이 무엇을 생각하고 있는지 설명합니다
2. **행동(Act)**: LLM이 도구를 호출합니다
3. **관찰(Observe)**: 도구가 결과를 반환합니다
4. **반복(Repeat)**: 작업이 끝날 때까지 반복합니다

## 사용할 도구

- **MCP Calendar Server**: 일정 관리(앞에서 사용한 9개의 캘린더 도구)
- **RAG 시스템**: 문서에서 정보 검색
- **계산기**: 간단한 산술 연산

에이전트는 사용자 요청에 따라 어떤 도구를 어떤 순서로 사용할지 스스로 결정합니다!

## 설정: 의존성 설치하기

In [ ]:
!pip3 install -q mcp httpx

Llama Stack 엔드포인트와 Calendar MCP Server 엔드포인트를 정의합니다.

In [ ]:
from llama_stack_client import LlamaStackClient
from llama_stack_client.lib.inference.event_logger import EventLogger
from mcp import ClientSession
from mcp.client.sse import sse_client
import json
from typing import Dict, Any, List
import re
import logging
from datetime import datetime

# HTTP 로그 출력 줄이기
logging.getLogger("httpx").setLevel(logging.WARNING)

# Llama Stack 서버에 연결
base_url = "http://llama-stack-service:8321"
client = LlamaStackClient(base_url=base_url)
model = "vllm-llama32/llama32"

# MCP 서버 URL
mcp_server_url = "http://canopy-mcp-calendar-mcp-server:8080/sse"

print(f"✅ Connected to Llama Stack at: {base_url}")
print(f"✅ Using model: {model}")
print(f"✅ MCP Server: {mcp_server_url}")

## 도구 설정하기

세 가지 도구 범주(계산기, MCP 캘린더, RAG)에 대한 래퍼 함수를 만들어 보겠습니다.

### 도구 1: 계산기

In [ ]:
def calculator_tool(operation: str, a: float, b: float) -> str:
    """기본 사칙연산을 수행합니다."""
    try:
        if operation == "add":
            result = a + b
        elif operation == "subtract":
            result = a - b
        elif operation == "multiply":
            result = a * b
        elif operation == "divide":
            if b == 0:
                return "Error: Cannot divide by zero"
            result = a / b
        else:
            return f"Error: Unknown operation '{operation}'"
        return f"Result: {result}"
    except Exception as e:
        return f"Error: {str(e)}"

print("✅ Calculator tool ready")

### 도구 2: MCP 캘린더 도구

In [ ]:
async def call_mcp_tool(tool_name: str, arguments: Dict[str, Any]) -> str:
    """MCP 캘린더 도구를 호출하고 결과를 문자열로 반환합니다."""
    try:
        async with sse_client(mcp_server_url) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                result = await session.call_tool(tool_name, arguments=arguments)
                return "\n".join([content.text for content in result.content])
    except Exception as e:
        return f"Error calling MCP tool: {str(e)}"

async def get_mcp_tools_description() -> str:
    """사용 가능한 모든 MCP 도구의 설명을 가져옵니다."""
    async with sse_client(mcp_server_url) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            
            descriptions = []
            for tool in tools.tools:
                descriptions.append(f"  - mcp_{tool.name}: {tool.description}")
            
            return "\n".join(descriptions)

# 사용 가능한 MCP 도구를 가져와 표시
mcp_tools_desc = await get_mcp_tools_description()
print("✅ MCP Calendar tools available:")
print(mcp_tools_desc)

### 도구 3: RAG(문서 검색)

에이전트가 문서를 검색할 수 있도록 RAG 시스템을 설정하겠습니다.

In [ ]:
# vector store가 이미 있는지 확인하고, 없으면 새로 만듦
try:
    # 기존 vector store 목록 조회
    vector_stores = client.vector_stores.list()
    
    if vector_stores.data and len(vector_stores.data) > 0:
        # 사용 가능한 첫 번째 vector store 사용
        vector_store = vector_stores.data[0]
        print(f"✅ Using existing vector store: {vector_store.id}")
    else:
        raise Exception("No vector store exists")
    
    # 정상 상태인지 확인하기 위한 테스트 질의
    query = "canopy"
    max_results = 5
    search_results = client.vector_stores.search(
        vector_store_id=vector_store.id,
        query=query,
        max_num_results=max_results,
        search_mode="vector"
    )
    print(f"search_results: {search_results.data}")

    if vector_store.file_counts.total > 0 and not search_results.data:
        client.vector_stores.delete(vector_store_id=vector_store.id)
        vector_store = client.vector_stores.create(
            name="my_citations_db",
            extra_body={
                "embedding_model": "sentence-transformers/nomic-ai/nomic-embed-text-v1.5",
                "embedding_dimension": 384,
                "provider_id": "milvus",
                "vector_db_id": "test"
            }
        )
except Exception as e:
    print(f"⚠️  Could not set up vector store: {e}")
    vector_store = None

In [ ]:
def rag_search_tool(query: str, max_results: int = 3) -> str:
    """RAG로 문서를 검색하고 관련 정보를 반환합니다."""
    if not vector_store:
        return "Error: RAG system not initialized. Please set up a vector store first."
    
    try:
        # 벡터 데이터베이스 검색
        search_results = client.vector_stores.search(
            vector_store_id=vector_store.id,
            query=query,
            max_num_results=max_results,
            search_mode="vector"
        )
        
        if not search_results.data:
            return "No relevant documents found."
        
        # 결과 정리
        results = []
        for i, result in enumerate(search_results.data, 1):
            content = result.content if hasattr(result, 'content') else str(result)
            results.append(f"Result {i}: {content}")
        
        return "\n\n".join(results)
    except Exception as e:
        return f"Error performing RAG search: {str(e)}"

print("✅ RAG search tool ready")

## ReAct 에이전트 만들기

이제 사용할 도구를 스스로 선택하는 ReAct 에이전트를 만들어 보겠습니다!

In [ ]:
def create_react_system_prompt() -> str:
    """ReAct 에이전트용 시스템 프롬프트를 만듭니다."""
    
    return f"""You are a helpful assistant that can use tools to answer questions.
The current date is {datetime.today().strftime('%Y-%m-%d')}

Available tools:

1. calculator
   - Performs arithmetic operations (add, subtract, multiply, divide)
   - Input format: {{"operation": "add"|"subtract"|"multiply"|"divide", "a": number, "b": number}}

2. MCP Calendar Tools:
{mcp_tools_desc}
   - For calendar tools, use format: {{"tool_name": "mcp_TOOLNAME", "arguments": {{...}}}}

3. rag_search
   - Searches through knowledge base documents for information
   - Input format: {{"query": "your search query", "max_results": 3}}
   - Use this when the user asks about concepts, definitions, or information not in the calendar

Use this format for your responses:

Thought: [Explain your reasoning about what to do next]
Action: [tool_name]
Action Input: {{"key": "value"}}

After receiving an observation, you can either:
1. Continue with another Thought/Action/Action Input if you need more information
2. Provide a final answer with: Final Answer: [your answer]

Important:
- Always explain your thinking in the Thought section
- Choose the most appropriate tool for the task
- For calendar operations, use the MCP tools (mcp_get_upcoming_events, mcp_create_event, etc.)
- For knowledge questions, use rag_search
- For calculations, use calculator
"""

print("✅ ReAct system prompt created")

## ReAct 루프 구현

In [ ]:
def parse_react_response(response: str) -> Dict[str, Any]:
    """ReAct 형식으로 된 LLM의 응답을 파싱합니다."""
    result = {}
    
    # Thought 추출
    thought_match = re.search(r'Thought:\s*(.+?)(?=\n(?:Action|Final Answer):|$)', response, re.DOTALL)
    if thought_match:
        result['thought'] = thought_match.group(1).strip()
    
    # Final Answer가 있는지 확인
    final_answer_match = re.search(r'Final Answer:\s*(.+)', response, re.DOTALL)
    if final_answer_match:
        result['final_answer'] = final_answer_match.group(1).strip()
        return result
    
    # Action 추출
    action_match = re.search(r'Action:\s*(.+?)(?=\n|$)', response)
    if action_match:
        result['action'] = action_match.group(1).strip()
    
    # Action Input 추출
    action_input_match = re.search(r'Action Input:\s*({.+})', response, re.DOTALL)
    if action_input_match:
        try:
            json_str = action_input_match.group(1).strip()
            # 흔한 JSON 형식 문제 정리
            json_str = re.sub(r'\n\s*', ' ', json_str)  # 줄바꿈 제거
            result['action_input'] = json.loads(json_str)
        except json.JSONDecodeError as e:
            result['error'] = f"Failed to parse action input: {e}"
    
    return result


async def execute_tool(action: str, action_input: Dict[str, Any]) -> str:
    """action에 맞는 도구를 실행합니다."""
    
    if action == "calculator":
        return calculator_tool(
            action_input.get("operation"),
            action_input.get("a"),
            action_input.get("b")
        )
    
    elif action.startswith("mcp_"):
        # MCP 캘린더 도구
        tool_name = action[4:]  # 'mcp_' 접두사 제거
        arguments = action_input.get("arguments", action_input)
        return await call_mcp_tool(tool_name, arguments)
    
    elif action == "rag_search":
        return rag_search_tool(
            action_input.get("query"),
            action_input.get("max_results", 3)
        )
    
    else:
        return f"Error: Unknown tool '{action}'"


async def run_react_agent(user_question: str, max_iterations: int = 5, verbose: bool = True):
    """ReAct 에이전트 루프를 실행합니다."""
    
    if verbose:
        print("="*70)
        print(f"USER QUESTION: {user_question}")
        print("="*70)
    
    system_prompt = create_react_system_prompt()
    conversation_history = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_question}
    ]
    
    for iteration in range(max_iterations):
        if verbose:
            print(f"\n{'='*70}")
            print(f"ITERATION {iteration + 1}")
            print("="*70)
        
        # LLM 응답 받기
        response = client.chat.completions.create(
            model=model,
            messages=conversation_history,
            stream=False,
        )
        
        llm_output = response.choices[0].message.content
        
        if verbose:
            print(f"\n{llm_output}")
        
        # 응답 파싱
        parsed = parse_react_response(llm_output)
        
        # 최종 답변이 있는지 확인
        if 'final_answer' in parsed:
            if verbose:
                print("\n" + "="*70)
                print("FINAL ANSWER:")
                print("="*70)
                print(parsed['final_answer'])
                print("="*70)
            return parsed['final_answer']
        
        # action이 있으면 실행
        if 'action' in parsed and 'action_input' in parsed:
            action = parsed['action']
            action_input = parsed['action_input']
            
            if verbose:
                print(f"\n🔧 Executing: {action}")
                print(f"📥 Input: {json.dumps(action_input, indent=2)}")
            
            # 도구 실행
            tool_result = await execute_tool(action, action_input)
            
            if verbose:
                print(f"\n📊 Observation:")
                result_preview = tool_result[:300] + "..." if len(tool_result) > 300 else tool_result
                print(result_preview)
            
            # 관찰 결과를 대화에 추가
            observation_text = f"Observation: {tool_result}"
            conversation_history.append({"role": "assistant", "content": llm_output})
            conversation_history.append({"role": "user", "content": observation_text})
        else:
            # LLM이 형식을 제대로 따르지 않음
            error_msg = "Please follow the Thought/Action/Action Input format or provide a Final Answer."
            if verbose:
                print(f"\n⚠️  {error_msg}")
            conversation_history.append({"role": "assistant", "content": llm_output})
            conversation_history.append({"role": "user", "content": error_msg})
    
    if verbose:
        print("\n" + "="*70)
        print("⚠️  WARNING: Max iterations reached without final answer")
        print("="*70)
    
    return "I couldn't find an answer within the iteration limit."

print("✅ ReAct agent is ready!")

## 에이전트 테스트하기!

이제 에이전트가 실제로 동작하는 모습을 보겠습니다. 각 작업에 맞는 도구를 스스로 고르는 과정을 지켜보세요.

### 예제 1: 계산기 사용하기

에이전트는 이 작업에 계산기가 필요하다는 것을 알아차려야 합니다.

In [ ]:
await run_react_agent("What is 847 multiplied by 923?")

### 예제 2: MCP 캘린더 사용하기

에이전트는 캘린더 MCP 도구를 사용해야 합니다.

In [ ]:
await run_react_agent("What events do I have coming up in the next 7 days?")

### 예제 3: RAG 검색 사용하기

에이전트는 RAG로 문서를 검색할 수 있으며, 이를 Agentic RAG라고도 부릅니다.  

**참고**: 제대로 동작하지 않는다면 `5-rag/2-intro-to-RAG.ipynb`를 실행해 canopy 프로젝트의 RAG 데이터베이스에 Canopy 문서를 수집했는지 확인하세요.

In [ ]:
await run_react_agent("What is the structure of Canopy in botany?")

### 예제 4: 여러 단계의 추론

이 작업을 하려면 에이전트가 다음을 수행해야 합니다.
1. 캘린더에서 일정 검색
2. 계산 수행

In [ ]:
await run_react_agent("How many events do I have in the next week, multiplied by 2?")

### 예제 5: 어려운 요청

에이전트에게 조금 더 어려운 요청 몇 가지를 해 봅니다.  
실행해 보고 에이전트가 해내는지 확인하세요. 해내지 못한다면 왜 실패했다고 생각하나요?

In [ ]:
await run_react_agent("Tell me about my schedule and what Canopy means")

In [ ]:
await run_react_agent("Create a meeting for tomorrow at 2pm called 'Team Sync'")

## 실제 환경에서의 시사점

에이전트 방식은 환상적으로 보이고 실제로도 그렇지만, 에이전트 워크플로를 사용할 때 알아 두어야 할 단점도 있습니다.

- **비용**: 반복 1회 = LLM 호출 1회 = $$$
- **지연 시간**: 추론이 많을수록 응답이 느려집니다
- **신뢰성**: 컨텍스트가 많아지므로 에이전트가 실수하기 쉬워지며, 특히 도구가 많을 때 그렇습니다
- **디버깅**: 여러 단계의 추론은 디버깅하기 어렵습니다

## 직접 해 보세요!

이제 여러분 차례입니다. 다음과 같은 질문을 시도해 보세요.
- 여러 다른 도구가 필요한 질문
- 어떤 도구를 사용할지 추론해야 하는 질문
- 캘린더, 지식, 계산을 결합해야 하는 질문

에이전트가 얼마나 잘 처리하는지 확인해 보세요!

In [ ]:
# 여러분 차례입니다! 직접 질문해 보세요:
await run_react_agent("YOUR QUESTION HERE")